# ML Cluster is required for this Notebook

## Prepare the Training Dataset

In [0]:
from pyspark.sql.functions import col, when, lit

# Load the Gold feature table
gold = spark.table("raghavan_trading_signals.gold.daily_features")

print(f"Total rows: {gold.count():,}")
print(f"Total columns: {len(gold.columns)}")

## Identify which columns are features vs metadata vs target

In [0]:
metadata_cols = [
    "symbol", "trade_date", "open", "high", "low", "close", "adj_close",
    "volume", "dividends", "stock_splits", "prev_close",
    "bronze_ingested_at", "bronze_source_file",
    "next_day_return",
]

target_col = "next_day_direction"

# Feature columns = everything that's not metadata and not the target
feature_cols = [
    c for c in gold.columns
    if c not in metadata_cols and c != target_col
]

print(f"Feature columns ({len(feature_cols)}):")
for i, c in enumerate(feature_cols):
    print(f"  {i+1}. {c}")

## Create the ML-ready dataset
Keep symbol and trade_date for reference, but tell AutoML to exclude them

In [0]:
ml_dataset = gold.select(
    "symbol",
    "trade_date",
    *feature_cols,
    target_col
).na.drop()

print(f"ML dataset: {ml_dataset.count():,} rows, {len(ml_dataset.columns)} columns")

ml_dataset.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("raghavan_trading_signals.ml.automl_training_data")

## Time-Based Train/Test Split
Why time-based and not random? In stock data, the future must never leak into the past. If you randomly split, your model might train on 2025 data and test on 2023 data — that's cheating. Always split by time: train on older data, test on newer data.

In [0]:
# Use the last 3 months as test set, everything before as training
from pyspark.sql.functions import max as spark_max

max_date = gold.agg(spark_max("trade_date")).collect()[0][0]
print(f"Latest date in dataset: {max_date}")

# ~63 trading days ≈ 3 months
split_date = "2025-01-01"

train_df = ml_dataset.filter(col("trade_date") < split_date)
test_df = ml_dataset.filter(col("trade_date") >= split_date)

print(f"Training set: {train_df.count():,} rows (before {split_date})")
print(f"Test set:     {test_df.count():,} rows (from {split_date} onward)")

# Save the splits
train_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("raghavan_trading_signals.ml.train_data")

test_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("raghavan_trading_signals.ml.test_data")

## Run AutoML

In [0]:
import databricks.automl as automl

exclude_cols = ["symbol", "trade_date"]

summary = automl.classify(
    dataset="raghavan_trading_signals.ml.train_data",
    target_col="next_day_direction",
    exclude_cols=exclude_cols,
    primary_metric="f1",
    timeout_minutes=30,
    experiment_name="trading-signals-automl",
)

##  Inspect the Results
AutoML returns a summary object with the best trial info

In [0]:
print(f"Best trial metric (F1): {summary.best_trial.metrics['val_f1_score']:.4f}")
print(f"Best trial model: {summary.best_trial.model_description}")

# The experiment is in MLflow — let's look at it
print(f"\nMLflow experiment: {summary.experiment.name}")
print(f"MLflow experiment ID: {summary.experiment.experiment_id}")

# The best model's run ID — we'll need this later
best_run_id = summary.best_trial.mlflow_run_id
print(f"Best run ID: {best_run_id}")

Load and evaluate the best model on the TEST set (data it's never seen)

In [0]:
import mlflow
from sklearn.metrics import classification_report, accuracy_score

# Load the best model
best_model_uri = f"runs:/{best_run_id}/model"
best_model = mlflow.sklearn.load_model(best_model_uri)

# Prepare test data as Pandas (AutoML trains sklearn-compatible models)
test_pdf = test_df.drop("symbol", "trade_date").toPandas()
X_test = test_pdf.drop("next_day_direction", axis=1)
y_test = test_pdf["next_day_direction"]

# Predict
y_pred = best_model.predict(X_test)

# Evaluate
print("=== Test Set Performance ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Down', 'Up'])}")

## Log Test Metrics to the Best Run

In [0]:

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("test_f1",
        float(classification_report(y_test, y_pred, output_dict=True)["weighted avg"]["f1-score"])
    )
    mlflow.set_tag("evaluation", "test_set_evaluated")
    print("Test metrics logged to MLflow run.")